# W9-E: compact mechanically grounded evidence (v1)

FIT-only controlled diagnostic: FIXED-5 / SEMANTIC@20, DraftSemantic, requested K=2. Primary K=0 remains frozen. A–D are read-only inputs. This notebook replaces E's previous free-form extraction interface; no F revision comparisons or new generator are introduced.

## 1. Protocol / frozen dependencies
Exact source support is mechanical, **not semantic entailment**. E1/E2 use a conservative copied-subspan fact variant. No prompt repair or downstream-F1 tuning is enabled. A preflight is mandatory before a full run.


In [1]:
from pathlib import Path
import sys, json, subprocess
import pandas as pd
from IPython.display import display
ROOT = Path.cwd().resolve()
while not (ROOT / 'src/week9').is_dir():
    if ROOT == ROOT.parent:
        raise FileNotFoundError('Open this notebook inside the project')
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.week9.grounded_e import prompts
from src.week9.grounded_e.protocol import config, output_dir, e_sources
from src.week9.grounded_e.runner import verify_gate
from src.week9.provenance import source_hashes, file_hash
CONFIG = ROOT / 'configs/week9/w9e_grounded.yaml'
cfg = config(CONFIG)
directory = output_dir(cfg)


## 2. Load W9-A/C/D artifacts
The CLI validates frozen A/B/C, complete D0–D3 coverage, D row/input hashes and canonical draft alignment. D1/D2/D3 must have identical chunk IDs **and text**. It creates only an E-local immutable D-input seal; no existing frozen artifact is rewritten.


In [2]:
seal_path = directory / 'protocol/d_input_freeze.json'
if seal_path.is_file():
    saved_seal = json.loads(seal_path.read_text())
    display({k: saved_seal[k] for k in ['frozen_retrieval', 'selector', 'evidence_k', 'primary_policy', 'num_queries']})
else:
    print('The first preflight will validate D and pin its read-only input seal.')


{'frozen_retrieval': {'representation': 'FIXED-5',
  'retriever': 'SEMANTIC',
  'retrieval_width': 20},
 'selector': 'DraftSemantic',
 'evidence_k': 2,
 'primary_policy': 'K_E=0',
 'num_queries': 1746}

## 3. Assert source identity
No candidate is regenerated or substituted. Exact offsets reconstruct canonical source text. Each E row records D row hashes, chunk text hashes, model/tokenizer/prompt/budget identities, evidence and output hashes.


In [3]:
if seal_path.is_file() and saved_seal['upstream_sources'] != source_hashes():
    raise ValueError('Historical A-D source bundle changed; do not reuse this protocol.')
print('Source identity is checked again against actual upstream files before any GPU call.')


Source identity is checked again against actual upstream files before any GPU call.


## 4. W9-E configuration
Default final evidence budget: 128 tokens **including IDs, JSON keys and formatting**. Alternative 64-token config: `configs/week9/w9e_grounded_64.yaml`, with separate artifacts and its own preflight. Intermediate LM extraction cap: 256 tokens; answer cap: unchanged 128. No silent cap increases.

First run: RUN_SMOKE=True, RUN_FULL=False. Review section 10. Only after review, enable RUN_FULL=True. If quality warnings exist, explicitly set ACK_PREFLIGHT_WARNINGS=True to acknowledge them; do not repeatedly tune prompts for F1.


In [4]:
RUN_FULL = True
RUN_SMOKE = False
ACK_PREFLIGHT_WARNINGS = True
SPLIT = 'fit'
assert SPLIT == 'fit'
base_command = [str(ROOT / 'sara_env/bin/python'), '-m', 'src.week9.grounded_e.runner',
                '--config', str(CONFIG), '--resume']
print('Evidence budget:', cfg['max_evidence_tokens'], '| intermediate cap:', cfg['extraction_max_tokens'])


Evidence budget: 128 | intermediate cap: 256


## 5. Smoke / preflight
Exactly three deterministic FIT queries are selected by a fixed seed/hash rule, without looking at gold outcomes. All four policies are checked. Full runs remain blocked without a current preflight matching sources, prompts, model/config identities and evidence budget. Preflight F1 is not used for policy selection.


In [5]:
if RUN_SMOKE:
    subprocess.run(base_command + ['--stage', 'preflight'], cwd=ROOT, check=True)
else:
    print('Preflight generation disabled. Existing saved preflight diagnostics are shown below if available.')


Preflight generation disabled. Existing saved preflight diagnostics are shown below if available.


## 6. E0 — compact extractive spans
Deterministic query-only BM25 ranking of sentences within the **already selected** chunks. A greedy packer copies exact word-boundary source prefixes, with source offsets, into the whole evidence budget. No extraction LM call and no generated facts. This is a relevance heuristic, not an oracle for sufficient evidence.


In [6]:
print(prompts.E0_DEFINITION)


Deterministic query-only BM25 ranking of source sentences; greedy exact-offset spans packed into the final evidence-token budget. No LM extraction or generated facts.


## 7. E1-Q — query-only structured grounding
One joint extraction call conditions on question and the selected source text, without the draft. Only strict JSON is accepted; unsupported spans, invented facts and oversized items are rejected. The final fact must be a short copied subspan of its valid support.


In [7]:
print(prompts.COMMON.replace('{budget}', str(cfg['max_evidence_tokens'])))
print(prompts.STRUCTURED)


Extract compact evidence from the provided sources. Sources are authoritative; never invent information. Return ONLY a JSON array, at most two objects. No markdown, preamble, rationale, explanation, or chain-of-thought. Prefer one short item directly answering the question. If no useful supported item exists, return [].
Copy support_span verbatim from its named source (at most 24 tokens). Copy fact or normalized_claim as a short contiguous phrase FROM THAT SUPPORT SPAN (at most 12 tokens); only whitespace normalization is allowed. Do not paraphrase or add claims. Use the exact source_chunk_id. Do not use the draft as evidence. Keep the entire representation, including keys and IDs, within 128 tokens. Invalid items will be rejected.
Schema: [{"source_chunk_id":"COPY_ID","support_span":"EXACT_QUOTE","fact":"SHORT_PHRASE_COPIED_FROM_QUOTE"}]. No additional fields.


## 8. E1-QD — draft-aware structured grounding
Same schema and budgets as E1-Q. The draft is provided solely to identify missing or conflicting source information. E1-QD minus E1-Q is the preregistered draft-conditioning comparison.


In [8]:
print(prompts.DRAFT)


The initial draft may be incomplete or wrong. Use it only to identify missing, uncertain, or conflicting information. Extract source evidence relevant to correcting or completing the answer. Do not preserve unsupported draft claims. Source evidence, not the draft, is authoritative.


## 9. E2 — compact grounded blueprint
The same verified support/short copied claim, plus minimal enumerated type and relevance labels. No rationales or chain-of-thought. Labels do not establish semantic correctness.


In [9]:
print(prompts.BLUEPRINT)


Use a minimal source-grounded blueprint, not a verbose table.
Schema: [{"source_chunk_id":"COPY_ID","support_span":"EXACT_QUOTE","normalized_claim":"SHORT_PHRASE_COPIED_FROM_QUOTE","evidence_type":"answer_detail","relevance":"direct"}].
Allowed evidence_type labels: answer_detail, correction, qualification, definition. Allowed relevance labels: direct, missing, conflicting. These are brief organizational labels, not extra factual statements. No additional fields.


## 10. Evidence-quality diagnostics and preflight gate
Review malformed, empty, fail-closed and cap-hit rates; proposed and accepted exact support; mean/median/p90 evidence tokens; compression; and accepted/rejected items. Source examples show canonical substrings and offsets. Failures return the saved draft **without calling the answer generator**. No raw-chunk or synthetic-evidence fallback exists.


In [10]:
preflight = directory / 'preflight'
if (preflight / 'preflight_gate.json').is_file():
    gate = json.loads((preflight / 'preflight_gate.json').read_text())
    display(pd.DataFrame(gate['quality']))
    for warning in gate['warnings']:
        print('WARNING:', warning)
    examples = json.loads((preflight / 'evidence_examples.json').read_text())
    for example in examples:
        display(example)
else:
    print('No completed preflight yet. Set RUN_SMOKE=True before attempting the full experiment.')
if RUN_FULL:
    identity_path = directory / 'protocol/grounded_e_identity.json'
    if not identity_path.is_file():
        raise ValueError('Complete the preflight first.')
    protocol_identity = json.loads(identity_path.read_text())
    if (protocol_identity['config'] != cfg or protocol_identity['e_sources'] != e_sources()
            or protocol_identity['upstream_sources'] != source_hashes()):
        raise ValueError('Saved preflight does not match the current configuration/source bundle.')
    verify_gate(directory, protocol_identity, ACK_PREFLIGHT_WARNINGS)


,policy,queries,exact_span_support_rate,accepted_exact_span_support_rate,mean_evidence_tokens,median_evidence_tokens,p90_evidence_tokens,compression_ratio,empty_extraction,invalid_format,fail_closed,reached_extraction_token_limit,zero_exact_support,num_supported_items,num_accepted_items,num_rejected_items,fact_tokens,support_tokens,num_accepted_facts,num_rejected_facts
0,E0,3,1.000000,1.0,96.666667,100.0,108.0,0.045064,0.0,0.000000,0.000000,0.000000,0.000000,2.000000,2.000000,0.000000,0.0,48.333333,0.000000,0.000000
1,E1-Q,3,0.333333,1.0,15.000000,0.0,36.0,0.007109,0.0,0.000000,0.666667,0.000000,0.666667,0.333333,0.333333,0.666667,1.0,1.000000,0.333333,0.666667
2,E1-QD,3,0.000000,NaN,0.000000,0.0,0.0,0.000000,0.0,0.666667,1.000000,0.000000,1.000000,0.000000,0.000000,0.333333,0.0,0.000000,0.000000,0.333333
3,E2,3,0.000000,NaN,0.000000,0.0,0.0,0.000000,0.0,0.666667,1.000000,0.333333,1.000000,0.000000,0.000000,0.333333,0.0,0.000000,0.000000,0.333333


{'question_id': '2038',
 'policy': 'E0',
 'evidence': {'text': '[2d82490f2fc7e53780cd88ce] An overview of the provided dataset is given in Table TABREF16 .\n[b490e4a32a6f10cdcfc2978b] Overall results between the monolingual and multilingual model are close, but the advantage of our multilingual All-in-1 approach is that it is a single model that can be applied to all four languages.',
  'items': [{'source_chunk_id': '2d82490f2fc7e53780cd88ce',
    'source_paper_id': '700',
    'question_id': '2038',
    'char_start': 72,
    'char_end': 136,
    'support_span': 'An overview of the provided dataset is given in Table TABREF16 .',
    'exact_source_text': 'An overview of the provided dataset is given in Table TABREF16 .',
    'support_valid': True,
    'support_tokens': 16,
    'token_count': 16},
   {'source_chunk_id': 'b490e4a32a6f10cdcfc2978b',
    'source_paper_id': '700',
    'question_id': '2038',
    'char_start': 2900,
    'char_end': 3101,
    'support_span': 'Overall results bet

{'question_id': '2038',
 'policy': 'E1-Q',
 'evidence': {'text': '[{"source_chunk_id":"2d82490f2fc7e53780cd88ce","support_span":"meaningless","fact":"meaningless"}]',
  'items': [{'source_chunk_id': '2d82490f2fc7e53780cd88ce',
    'source_paper_id': '700',
    'question_id': '2038',
    'char_start': 0,
    'char_end': 11,
    'support_span': 'meaningless',
    'exact_source_text': 'meaningless',
    'support_valid': True,
    'support_tokens': 3,
    'token_count': 3,
    'fact': 'meaningless',
    'fact_tokens': 3,
    'fact_support_mode': 'copied_contiguous_subspan_whitespace_normalized'}],
  'rejected_items': [],
  'evidence_tokens': 45,
  'support_tokens': 3,
  'fact_tokens': 3,
  'num_supported_items': 1,
  'num_accepted_items': 1,
  'num_rejected_items': 0,
  'num_accepted_facts': 1,
  'num_rejected_facts': 0,
  'exact_span_support_rate': 1.0,
  'accepted_exact_span_support_rate': 1.0,
  'zero_exact_support': False,
  'invalid_format': False,
  'empty_extraction': False,
  'fail

{'question_id': '2038',
 'policy': 'E1-QD',
 'evidence': {'text': '',
  'items': [],
  'rejected_items': [{'reason': 'unsupported_span_or_source',
    'proposed': {'source_chunk_id': 'b490e4a32a6f10cdcfc2978b',
     'support_span': 'the final label distribution for the data is given in Figure FIGREF17 ',
     'fact': 'final label distribution for the data is given in Figure FIGREF17 '}}],
  'evidence_tokens': 0,
  'support_tokens': 0,
  'fact_tokens': 0,
  'num_supported_items': 0,
  'num_accepted_items': 0,
  'num_rejected_items': 1,
  'num_accepted_facts': 0,
  'num_rejected_facts': 1,
  'exact_span_support_rate': 0.0,
  'accepted_exact_span_support_rate': None,
  'zero_exact_support': True,
  'invalid_format': False,
  'empty_extraction': False,
  'fail_closed': True,
  'failure_reason': 'all_items_rejected_or_over_budget',
  'extraction_valid': False,
  'reached_extraction_token_limit': False,
  'evidence_hash': '12ae32cb1ec02d01eda3581b127c1fee3b0dc53572ed6baf239721a03d82e126',
  

{'question_id': '2038',
 'policy': 'E2',
 'evidence': {'text': '',
  'items': [],
  'rejected_items': [],
  'evidence_tokens': 0,
  'support_tokens': 0,
  'fact_tokens': 0,
  'num_supported_items': 0,
  'num_accepted_items': 0,
  'num_rejected_items': 0,
  'num_accepted_facts': 0,
  'num_rejected_facts': 0,
  'exact_span_support_rate': None,
  'accepted_exact_span_support_rate': None,
  'zero_exact_support': True,
  'invalid_format': True,
  'empty_extraction': False,
  'fail_closed': True,
  'failure_reason': 'malformed_output',
  'extraction_valid': False,
  'reached_extraction_token_limit': True,
  'evidence_hash': '12ae32cb1ec02d01eda3581b127c1fee3b0dc53572ed6baf239721a03d82e126',
  'max_evidence_tokens': 128,
  'grounding_note': 'Exact source support is mechanical, not semantic entailment. Facts are conservatively copied subspans, not free paraphrases.'},
 'support_verification': []}

...

{'question_id': '1583',
 'policy': 'E2',
 'evidence': {'text': '',
  'items': [],
  'rejected_items': [],
  'evidence_tokens': 0,
  'support_tokens': 0,
  'fact_tokens': 0,
  'num_supported_items': 0,
  'num_accepted_items': 0,
  'num_rejected_items': 0,
  'num_accepted_facts': 0,
  'num_rejected_facts': 0,
  'exact_span_support_rate': None,
  'accepted_exact_span_support_rate': None,
  'zero_exact_support': True,
  'invalid_format': True,
  'empty_extraction': False,
  'fail_closed': True,
  'failure_reason': 'malformed_output',
  'extraction_valid': False,
  'reached_extraction_token_limit': False,
  'evidence_hash': '12ae32cb1ec02d01eda3581b127c1fee3b0dc53572ed6baf239721a03d82e126',
  'max_evidence_tokens': 128,
  'grounding_note': 'Exact source support is mechanical, not semantic entailment. Facts are conservatively copied subspans, not free paraphrases.'},
 'support_verification': []}

{'question_id': '567',
 'policy': 'E0',
 'evidence': {'text': "[dc4a521868372dacca2f219f] For the SR and HATE datasets, the authors reported their trained best logistic regression model's results on their respective datasets.\n[4093084f447b7849e57bda19] in what way?",
  'items': [{'source_chunk_id': 'dc4a521868372dacca2f219f',
    'source_paper_id': '153',
    'question_id': '567',
    'char_start': 3116,
    'char_end': 3251,
    'support_span': "For the SR and HATE datasets, the authors reported their trained best logistic regression model's results on their respective datasets.",
    'exact_source_text': "For the SR and HATE datasets, the authors reported their trained best logistic regression model's results on their respective datasets.",
    'support_valid': True,
    'support_tokens': 27,
    'token_count': 27},
   {'source_chunk_id': '4093084f447b7849e57bda19',
    'source_paper_id': '153',
    'question_id': '567',
    'char_start': 753,
    'char_end': 765,
    'support_span'

{'question_id': '567',
 'policy': 'E1-Q',
 'evidence': {'text': '',
  'items': [],
  'rejected_items': [{'reason': 'unsupported_span_or_source',
    'proposed': {'source_chunk_id': 'dc4a521868372dacca2f219f',
     'support_span': 'F1: 0.8200',
     'fact': '0.8200'}}],
  'evidence_tokens': 0,
  'support_tokens': 0,
  'fact_tokens': 0,
  'num_supported_items': 0,
  'num_accepted_items': 0,
  'num_rejected_items': 1,
  'num_accepted_facts': 0,
  'num_rejected_facts': 1,
  'exact_span_support_rate': 0.0,
  'accepted_exact_span_support_rate': None,
  'zero_exact_support': True,
  'invalid_format': False,
  'empty_extraction': False,
  'fail_closed': True,
  'failure_reason': 'all_items_rejected_or_over_budget',
  'extraction_valid': False,
  'reached_extraction_token_limit': False,
  'evidence_hash': '12ae32cb1ec02d01eda3581b127c1fee3b0dc53572ed6baf239721a03d82e126',
  'max_evidence_tokens': 128,
  'grounding_note': 'Exact source support is mechanical, not semantic entailment. Facts are co

{'question_id': '567',
 'policy': 'E1-QD',
 'evidence': {'text': '',
  'items': [],
  'rejected_items': [],
  'evidence_tokens': 0,
  'support_tokens': 0,
  'fact_tokens': 0,
  'num_supported_items': 0,
  'num_accepted_items': 0,
  'num_rejected_items': 0,
  'num_accepted_facts': 0,
  'num_rejected_facts': 0,
  'exact_span_support_rate': None,
  'accepted_exact_span_support_rate': None,
  'zero_exact_support': True,
  'invalid_format': True,
  'empty_extraction': False,
  'fail_closed': True,
  'failure_reason': 'malformed_output',
  'extraction_valid': False,
  'reached_extraction_token_limit': False,
  'evidence_hash': '12ae32cb1ec02d01eda3581b127c1fee3b0dc53572ed6baf239721a03d82e126',
  'max_evidence_tokens': 128,
  'grounding_note': 'Exact source support is mechanical, not semantic entailment. Facts are conservatively copied subspans, not free paraphrases.'},
 'support_verification': []}

{'question_id': '567',
 'policy': 'E2',
 'evidence': {'text': '',
  'items': [],
  'rejected_items': [{'reason': 'unsupported_span_or_source',
    'proposed': {'source_chunk_id': 'dc4a521868372dacca2f219f',
     'support_span': '100k parameters',
     'normalized_claim': '100k parameters',
     'evidence_type': 'answer_detail',
     'relevance': 'direct'}}],
  'evidence_tokens': 0,
  'support_tokens': 0,
  'fact_tokens': 0,
  'num_supported_items': 0,
  'num_accepted_items': 0,
  'num_rejected_items': 1,
  'num_accepted_facts': 0,
  'num_rejected_facts': 1,
  'exact_span_support_rate': 0.0,
  'accepted_exact_span_support_rate': None,
  'zero_exact_support': True,
  'invalid_format': False,
  'empty_extraction': False,
  'fail_closed': True,
  'failure_reason': 'all_items_rejected_or_over_budget',
  'extraction_valid': False,
  'reached_extraction_token_limit': False,
  'evidence_hash': '12ae32cb1ec02d01eda3581b127c1fee3b0dc53572ed6baf239721a03d82e126',
  'max_evidence_tokens': 128,
  '

## 11. Downstream evaluation
The answer generator uses precisely the D regeneration behavior (revision=False), canonical context, latent embeddings and 128-token answer limit. Only the evidence object changes. This section does not test F0/F1/F2. All D rows are imported read-only, not regenerated. Full runs resume only compatible E caches.


In [11]:
if RUN_FULL:
    command = base_command + ['--stage', 'full']
    if ACK_PREFLIGHT_WARNINGS:
        command.append('--ack-preflight-warnings')
    subprocess.run(command, cwd=ROOT, check=True)
else:
    print('FULL W9-E RUN DISABLED. No full generations were started.')
fit_directory = directory / 'fit'
if (fit_directory / 'manifest.json').is_file():
    manifest = json.loads((fit_directory / 'manifest.json').read_text())
    print('Saved run:', manifest)
    display(pd.read_csv(fit_directory / 'answer_and_cost_summary.csv'))
else:
    print('Full FIT results do not exist yet; no fallback metrics are invented.')


{"preflight_quality": [{"policy": "E0", "queries": 3, "exact_span_support_rate": 1.0, "accepted_exact_span_support_rate": 1.0, "mean_evidence_tokens": 96.66666666666667, "median_evidence_tokens": 100.0, "p90_evidence_tokens": 108.0, "compression_ratio": 0.045064266825263405, "empty_extraction": 0.0, "invalid_format": 0.0, "fail_closed": 0.0, "reached_extraction_token_limit": 0.0, "zero_exact_support": 0.0, "num_supported_items": 2.0, "num_accepted_items": 2.0, "num_rejected_items": 0.0, "fact_tokens": 0.0, "support_tokens": 48.333333333333336, "num_accepted_facts": 0.0, "num_rejected_facts": 0.0}, {"policy": "E1-Q", "queries": 3, "exact_span_support_rate": 0.3333333333333333, "accepted_exact_span_support_rate": 1.0, "mean_evidence_tokens": 15.0, "median_evidence_tokens": 0.0, "p90_evidence_tokens": 36.0, "compression_ratio": 0.007109004739336493, "empty_extraction": 0.0, "invalid_format": 0.0, "fail_closed": 0.6666666666666666, "reached_extraction_token_limit": 0.0, "zero_exact_support

`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 3/3 [00:01<00:00,  2.33it/s]
Some weights of XMistralForCausalLM were not initialized from the model checkpoint at mistralai/Mistral-7B-Instruct-v0.2 and are newly initialized: ['projector.projector.0.bias', 'projector.projector.0.weight', 'projector.projector.2.bias', 'projector.projector.2.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
The new lm_head weights will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `m

W9-E full query 1/1746 512 E0
W9-E full query 1/1746 512 E1-Q
W9-E full query 1/1746 512 E1-QD
W9-E full query 1/1746 512 E2
W9-E full query 2/1746 513 E0
W9-E full query 2/1746 513 E1-Q
W9-E full query 2/1746 513 E1-QD
W9-E full query 2/1746 513 E2
W9-E full query 3/1746 745 E0
W9-E full query 3/1746 745 E1-Q
W9-E full query 3/1746 745 E1-QD
W9-E full query 3/1746 745 E2
W9-E full query 4/1746 746 E0
W9-E full query 4/1746 746 E1-Q
W9-E full query 4/1746 746 E1-QD
W9-E full query 4/1746 746 E2
W9-E full query 5/1746 1633 E0
W9-E full query 5/1746 1633 E1-Q
W9-E full query 5/1746 1633 E1-QD
W9-E full query 5/1746 1633 E2
W9-E full query 6/1746 1221 E0
W9-E full query 6/1746 1221 E1-Q
W9-E full query 6/1746 1221 E1-QD
W9-E full query 6/1746 1221 E2
W9-E full query 7/1746 1222 E0
W9-E full query 7/1746 1222 E1-Q
W9-E full query 7/1746 1222 E1-QD
W9-E full query 7/1746 1222 E2
W9-E full query 8/1746 1223 E0
W9-E full query 8/1746 1223 E1-Q
W9-E full query 8/1746 1223 E1-QD
W9-E full query

,policy,queries,f1,exact_match,delta_f1,help,hurt,answer_changed,final_prompt_tokens,total_query_prompt_tokens,total_query_generated_tokens,num_lm_calls,retrieval_query_time,selection_latency,extraction_latency,revision_latency,measured_end_to_end_latency,descriptive_help_over_help_plus_hurt
0,D0,1746,0.385631,0.227950,0.000000,0.000000,0.000000,0.000000,940.544101,940.544101,16.654639,1.000000,0.00496,8.662111e-07,NaN,NaN,NaN,NaN
1,D1,1746,0.385004,0.193013,-0.000627,0.321306,0.281787,0.762887,3041.357961,3981.902062,56.474800,2.000000,0.00496,2.936539e-01,NaN,2.261914,NaN,0.532764
2,D2,1746,0.368659,0.205040,-0.016972,0.210767,0.238832,0.646048,1476.545246,5618.391180,730.073310,3.000000,0.00496,2.932711e-01,27.692476,1.196957,NaN,0.468790
3,D3,1746,0.376936,0.211340,-0.008695,0.186712,0.201031,0.553265,1508.739404,5690.195876,639.876861,3.000000,0.00496,2.934705e-01,24.249098,1.162383,NaN,0.481536
4,E0,1746,0.366109,0.209049,-0.019522,0.178694,0.221077,0.596220,1093.055556,2033.599656,40.935853,2.000000,0.00496,2.936539e-01,0.005353,1.093965,NaN,0.446991
5,E1-Q,1746,0.382523,0.223940,-0.003108,0.010309,0.014891,0.040092,947.396907,3365.097938,167.587629,2.066438,0.00496,2.936539e-01,6.015287,0.889616,NaN,0.409091
6,E1-QD,1746,0.382099,0.226231,-0.003532,0.012027,0.017182,0.048110,948.475945,3455.739977,157.253723,2.076174,0.00496,2.936539e-01,5.686586,0.762657,NaN,0.411765
7,E2,1746,0.383591,0.225659,-0.002040,0.020619,0.020046,0.061283,955.730241,3590.909507,175.191294,2.127721,0.00496,2.936539e-01,6.362793,0.737052,NaN,0.507042


## 12. Cost accounting
Summary rows compare D0/D1/D2/D3 with E0/E1-Q/E1-QD/E2. Final prompt tokens, cumulative prompt/generated tokens and LM calls include the saved initial draft and every extraction/answer call. Retrieval and selector timings are inherited from the fixed D source selection. They are not generator input tokens. Cached component latencies are reported separately; end-to-end latency is unmeasured.


In [12]:
if (fit_directory / 'evidence_quality.csv').is_file():
    display(pd.read_csv(fit_directory / 'evidence_quality.csv'))
else:
    print('Evidence-quality and cost tables await a full run.')


,policy,queries,exact_span_support_rate,accepted_exact_span_support_rate,mean_evidence_tokens,median_evidence_tokens,p90_evidence_tokens,compression_ratio,empty_extraction,invalid_format,fail_closed,reached_extraction_token_limit,zero_exact_support,num_supported_items,num_accepted_items,num_rejected_items,fact_tokens,support_tokens,num_accepted_facts,num_rejected_facts
0,E0,1746,1.000000,1.0,105.619129,106.0,128.0,0.054740,0.000000,0.000000,0.000000,0.000000,0.000000,1.991409,1.990263,0.001145,0.000000,57.966208,0.000000,0.000000
1,E1-Q,1746,0.542446,1.0,3.782360,0.0,0.0,0.002106,0.000000,0.631157,0.933562,0.272050,0.797251,0.215922,0.068729,0.329324,0.399771,0.668958,0.068729,0.329324
2,E1-QD,1746,0.587200,1.0,4.387171,0.0,0.0,0.002201,0.003436,0.679840,0.923826,0.237686,0.806415,0.210195,0.080756,0.277205,0.475372,0.738832,0.080756,0.277205
3,E2,1746,0.580739,1.0,9.218213,0.0,65.0,0.004632,0.001145,0.443872,0.872279,0.284078,0.674685,0.341924,0.128866,0.459908,0.722222,1.093356,0.128866,0.459908


## 13. Bootstrap statistics
Deterministic 10,000 paper-cluster resamples for E0−D0, E0−D1, E1-Q−D0, E1-QD−D0, E1-QD−E1-Q, E1-QD−D3, E2−D0, E2−E1-QD. Report point estimates and intervals. HELP/(HELP+HURT) is descriptive only. Neither preflight F1 nor a tiny positive point estimate selects a method.


In [13]:
bootstrap_path = fit_directory / 'paired_paper_bootstrap.json'
if bootstrap_path.is_file():
    display(pd.DataFrame(json.loads(bootstrap_path.read_text())).T)
else:
    print('Paired inference awaits complete full-FIT coverage.')


,point_estimate,ci95,resamples,cluster,seed
E0 - D0,-0.019522,"[-0.03304016860942537, -0.0060434805369102215]",10000,paper_id,42
E0 - D1,-0.018895,"[-0.035091298008303407, -0.002403584841573546]",10000,paper_id,42
E1-Q - D0,-0.003108,"[-0.00665055395181964, 0.00030840565595074054]",10000,paper_id,42
E1-QD - D0,-0.003532,"[-0.007989492636660697, 0.0008129449347053326]",10000,paper_id,42
E1-QD - E1-Q,-0.000424,"[-0.004698303997627098, 0.0037320870889856477]",10000,paper_id,42
E1-QD - D3,0.005162,"[-0.007863748062041933, 0.01866377289154292]",10000,paper_id,42
E2 - D0,-0.00204,"[-0.0065447371425307165, 0.0022450706578063763]",10000,paper_id,42
E2 - E1-QD,0.001493,"[-0.0033457348851471625, 0.006176804383860907]",10000,paper_id,42


## 14. Final summary / interpretation
- E0 gains over both D0 and D1 with positive paired intervals support compact copied-span evidence in this protocol.
- E1-QD gains over E1-Q require a positive paired interval before claiming draft conditioning helps.
- E2 gains over E1-QD require paired evidence before claiming organization helps.
- Negative/inconclusive outcomes remain valid. High failure rates limit what they establish about usable compact evidence. Do not add variants until an F1 gain appears.

Primary K=0 stays frozen. No E winner, F run, G training, TEST evaluation or model/projector training is launched or frozen here.


In [14]:
report_path = fit_directory / 'REPORT.md'
if report_path.is_file():
    print(report_path.read_text())
else:
    print('No full-run scientific conclusion yet. Inspect the preflight before committing GPU time.')


# W9-E compact grounded evidence

Complete FIT diagnostic; not held-out confirmation.

Primary policy stays K=0. Exact-span support is mechanical, not semantic entailment. Facts use the conservative copied-subspan variant. No outcome-driven prompt tuning. All extraction calls are charged; cached initial generation is counted, with its latency unknown. End-to-end latency remains unmeasured.

No requested superiority conclusion is supported by positive paired intervals. Inconclusive intervals do not establish equivalence or failure.
E1-Q: malformed rate/value 0.631 exceeds preregistered warning 0.2.
E1-Q: fail-closed rate/value 0.934 exceeds preregistered warning 0.5.
E1-Q: generation-cap hits rate/value 0.272 exceeds preregistered warning 0.2.
E1-Q: proposed exact-span support below .8. Rejected evidence must never reach the generator.
E1-QD: malformed rate/value 0.680 exceeds preregistered warning 0.2.
E1-QD: fail-closed rate/value 0.924 exceeds preregistered warning 0.5.
E1-QD: genera